<a href="https://colab.research.google.com/github/anujithesh26-del/SIH-2026-AI-ML-ENABLED-SEAMLESS-NAVIGATION/blob/TASK--A-IO-VNB-DATASET-UNDERSTANDING-AND-PREPROCESSING/SIH_TRACKA_STEP_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import files # Import files for upload/download

# ---- config (edit these two lines to change windowing) ----
WINDOW_LENGTH_S = 5.0
STRIDE_S = 2.5
SESSION_ID = "VW10"  # only one session/drive exists in this file

# ---- Upload Input CSV ----
print("Please upload the 'cleaned_MERGED_SVW10-VW10.csv' file.")
uploaded = files.upload()
# Assuming only one file is uploaded, get its name
INPUT_CSV_NAME = next(iter(uploaded))
print(f"Uploaded file: {INPUT_CSV_NAME}")
INPUT_CSV = INPUT_CSV_NAME # Use the uploaded file name as input path

OUT_DIR = Path("./outputs") # Save outputs to current directory for easier download

# ---- load ----
df = pd.read_csv(INPUT_CSV)
df = df.sort_values("time_s").reset_index(drop=True)

# ---- derive sample rate from actual timestamps (not assumed) ----
dt = df["time_s"].diff().dropna()
sample_rate_hz = round(1.0 / dt.median(), 4)
window_len_samples = int(round(WINDOW_LENGTH_S * sample_rate_hz))
stride_samples = int(round(STRIDE_S * sample_rate_hz))
print(f"Measured sample rate: {sample_rate_hz} Hz")
print(f"Window: {WINDOW_LENGTH_S}s = {window_len_samples} samples, "
      f"stride: {STRIDE_S}s = {stride_samples} samples")

# ---- feature columns: all real, numeric, non-constant columns ----
exclude_cols = {
    "time_s", "s_time_corrected_s",                    # kept separately as timestamps
    "Sample period (seconds)",                          # constant, not a feature
    "S_GPS SATELLITES IN RANGE",                         # string ("23 / 23")
    "S_DATE (YYYY-MO-DD HH-MI-SS_SSS)",                  # string timestamp
    "S_TIME SINCE START (ms)",                           # redundant timestamp
    "Time Since Start of Day (seconds)",                 # redundant timestamp
}
feature_cols = [c for c in df.columns if c not in exclude_cols]
print(f"\n{len(feature_cols)} feature columns (all pre-existing, none invented):")
for c in feature_cols:
    print(" -", c)

# sanity: confirm every feature column is numeric
non_numeric = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(df[c])]
assert not non_numeric, f"Non-numeric feature columns found: {non_numeric}"

# ---- build windows ----
n_rows = len(df)
X, starts, ends, start_idx, gps_dropout_flag = [], [], [], [], []

i = 0
while i + window_len_samples <= n_rows:
    window = df.iloc[i:i + window_len_samples]
    X.append(window[feature_cols].to_numpy(dtype=np.float32))
    starts.append(window["time_s"].iloc[0])
    ends.append(window["time_s"].iloc[-1])
    start_idx.append(i)
    # flag windows that overlap the GPS-satellite dropout anomaly (NaNs in that column)
    gps_dropout_flag.append(bool(window["No of GPS Satellites Available"].isna().any()))
    i += stride_samples

X = np.stack(X)  # (n_windows, window_len_samples, n_features)
session_ids = np.array([SESSION_ID] * len(X))
window_start_s = np.array(starts, dtype=np.float64)
window_end_s = np.array(ends, dtype=np.float64)
start_row_idx = np.array(start_idx, dtype=np.int64)
gps_dropout_flag = np.array(gps_dropout_flag, dtype=bool)

print(f"\nGenerated {len(X)} windows, shape per window: "
      f"({window_len_samples} samples, {len(feature_cols)} features)")
print(f"Windows overlapping the GPS-dropout anomaly: {gps_dropout_flag.sum()} / {len(X)}")

# ---- export ----
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_npz = OUT_DIR / "vw10_windows.npz"
np.savez(
    out_npz,
    X=X,
    feature_names=np.array(feature_cols),
    session_id=session_ids,
    window_start_s=window_start_s,
    window_end_s=window_end_s,
    start_row_idx=start_row_idx,
    gps_dropout_flag=gps_dropout_flag,
    sample_rate_hz=np.array([sample_rate_hz]),
    window_length_s=np.array([WINDOW_LENGTH_S]),
    stride_s=np.array([STRIDE_S]),
)
print(f"\nSaved: {out_npz}")

# ---- Download Output NPZ ----
files.download(out_npz) # Download the output file

Please upload the 'cleaned_MERGED_SVW10-VW10.csv' file.


Saving cleaned_MERGED SVW10-VW10.csv to cleaned_MERGED SVW10-VW10.csv
Uploaded file: cleaned_MERGED SVW10-VW10.csv
Measured sample rate: 10.0 Hz
Window: 5.0s = 50 samples, stride: 2.5s = 25 samples

48 feature columns (all pre-existing, none invented):
 - No of GPS Satellites Available
 - Latitude (degrees)
 - Longitude (degrees)
 - Velocity (km/hr)
 - Heading (degrees)
 - Height (km)
 - Vertical velocity (km/hr)
 - Steering Angle (degrees)
 - Wheel Speed Front Left (rad/sec)
 - Wheel Speed Front Right (rad/sec)
 - Wheel Speed Rear Left (rad/sec)
 - Wheel Speed Rear Right (rad/sec)
 - Yaw Rate (deg/sec)
 - Indicated Vehicle Speed (km/hr)
 - Indicated Longitudinal Acceleration (g)
 - Indicated Lateral Acceleration (g)
 - Handbrake (0 or 1)
 - Gear Requested (Number fof gear employed 1-5)
 - Gear (Number fof gear employed 1-5)
 - Engine Speed (rev/min)
 - Coolant Temperature (degrees)
 - Clutch Position (0 or 1)
 - Brake Pressure (psi)
 - Brake Position (0 or 1)
 - Battery Voltage (volts

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>